# Tandem-repeat pathogenicity visualizer

Interactive exploration of the "population tightness x pathogenic distance" question for
your 63-locus catalog of known disease-associated tandem repeats.

**Data sources** (both bundled in `data/`, loaded fresh below — nothing here depends on a
precomputed table):

- `data/TR_catalog_63loci.json` — your uploaded catalog (`TRExplorerV1:KnownDiseaseAssociatedLoci`),
  with population allele histograms/SDs from HPRC256, TenK10K, T2T assemblies, and AoU1027.
- `data/STRchive_loci.json` — [STRchive](https://strchive.org) (Genome Medicine 2025), the source
  of pathogenic/benign/intermediate thresholds. Matched to your 63 loci by chromosome + coordinate
  + gene (all 63 resolve; `ENSG00000260851` matches BEAN1/SCA31 by coordinates).

**What you can play with:**
- **Population-SD cutoff** (tight vs. spread) and **pathogenic-distance cutoff** (close vs. far) —
  the two thresholds that define the four quadrants. Defaults are 5 repeat units and 3x fold-change,
  which is where the natural gaps fall in this dataset — not universal constants.
- **Distance metric**: raw fold-change (pathogenic_min / reference count) vs. z-score
  (pathogenic distance / population SD).
- **STRchive evidence tier** and **inheritance pattern** filters, to check whether the
  coding/noncoding pattern survives on just the well-established loci.

Run all cells, then use the controls under the plot. Everything recomputes live, including the
Fisher's-exact / Mann-Whitney stats below the figure.

**Setup:** `pip install -r requirements.txt` (see README). The plot redraws a plain
`plotly.graph_objects.Figure` on every slider/toggle change (rather than a live-patched
`FigureWidget`), which is slightly less smooth but avoids a version-compatibility break
some `FigureWidget` + `ipywidgets` + Jupyter combinations hit.


In [1]:
import json, re, math
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

DATA_DIR = Path("data")


C:\Users\nicol\AppData\Local\Temp\ipykernel_19688\3702761045.py:6: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.3.0)
  from scipy import stats


## 1. Build the per-locus table

Parses allele histograms (shift-invariant SD, pooled across whichever cohorts have data,
weighted by allele count), matches each locus to its STRchive disease-threshold entry, and
computes fold-change and z-score relative to the reference (haploid, healthy-genome) copy number.


In [2]:
def norm_chrom(c):
    return c.replace("chr", "")


def find_strchive_match(locus, strchive):
    # Match a catalog locus to its STRchive entry by chrom + gene + nearest coordinates.
    chrom = locus["ReferenceRegion"].split(":")[0].replace("chr", "")
    start0, end1 = int(locus["start_0based"]), int(locus["end_1based"])
    gene = locus["GencodeGeneName"]
    candidates = [s for s in strchive if norm_chrom(s["chrom"]) == chrom and s["gene"] == gene]
    if not candidates:
        candidates = [s for s in strchive if norm_chrom(s["chrom"]) == chrom]
    best, best_score = None, float("inf")
    for s in candidates:
        score = abs(s["start_hg38"] - start0) + abs(s["stop_hg38"] - end1)
        if score < best_score:
            best, best_score = s, score
    return best


def parse_hist(histstr):
    # '12x:300,13x:200' -> (n, mean, population_sd). SD is invariant to any additive
    # offset used by the encoding, so it's reliable even where the encoded mean isn't.
    if not histstr:
        return None
    total_n, total_sum, pairs = 0, 0.0, []
    for tok in histstr.split(","):
        tok = tok.strip()
        m = re.match(r"(-?\d+)x:(\d+)", tok)
        if not m:
            continue
        val, cnt = int(m.group(1)), int(m.group(2))
        pairs.append((val, cnt))
        total_n += cnt
        total_sum += val * cnt
    if total_n == 0:
        return None
    mean = total_sum / total_n
    var = sum(cnt * (val - mean) ** 2 for val, cnt in pairs) / total_n
    return (total_n, mean, math.sqrt(var))


HIST_FIELDS = [
    ("TenK10K", "TenK10K_AlleleHistogram"),
    ("HPRC256", "HPRC256_AlleleHistogram"),
    ("T2T", "AlleleFrequenciesFromT2TAssemblies"),
    ("Illumina174k", "AlleleFrequenciesFromIllumina174k"),
]


def classify_region(gencode_region):
    if gencode_region == "CDS":
        return "coding"
    if gencode_region == "exon":
        return "ambiguous"
    return "noncoding"


def build_dataframe(catalog_path=DATA_DIR / "TR_catalog_63loci.json",
                     strchive_path=DATA_DIR / "STRchive_loci.json"):
    catalog = json.load(open(catalog_path))
    strchive = json.load(open(strchive_path))

    rows = []
    for locus in catalog:
        strc = find_strchive_match(locus, strchive)
        if strc is None:
            continue

        cohorts = {}
        for name, field in HIST_FIELDS:
            h = locus.get(field)
            parsed = parse_hist(h) if h else None
            if parsed:
                cohorts[name] = parsed  # (n, mean, sd)

        if cohorts:
            total_n = sum(n for n, _, _ in cohorts.values())
            pooled_var = sum(n * sd ** 2 for n, _, sd in cohorts.values()) / total_n
            pooled_sd = math.sqrt(pooled_var)
            cohort_used = "+".join(sorted(cohorts.keys()))
        else:
            total_n, pooled_sd, cohort_used = 0, None, None

        ref = int(locus["NumRepeatsInReference"])
        path_min = strc.get("pathogenic_min")

        row = {
            "gene": locus["GencodeGeneName"],
            "disease_id": strc["id"],
            "disease": strc["disease"],
            "locus_id": locus["LocusId"],
            "motif": locus["ReferenceMotif"],
            "motif_len_bp": int(locus["MotifSize"]),
            "ref_repeats": ref,
            "region_catalog": locus.get("GencodeGeneRegion"),
            "region_class": classify_region(locus.get("GencodeGeneRegion")),
            "region_strchive": strc.get("location_in_gene"),
            "inheritance": strc.get("inheritance") or [],
            "evidence": (strc.get("evidence") or [None])[0],
            "benign_max": strc.get("benign_max"),
            "intermediate_min": strc.get("intermediate_min"),
            "intermediate_max": strc.get("intermediate_max"),
            "pathogenic_min": path_min,
            "pathogenic_max": strc.get("pathogenic_max"),
            "total_allele_n": total_n,
            "pooled_pop_sd": pooled_sd,
            "cohort_used": cohort_used,
            "per_cohort_sd": {k: round(v[2], 3) for k, v in cohorts.items()},
        }
        if path_min is not None and ref:
            row["fold_change"] = path_min / ref
            row["delta_units"] = path_min - ref
            if pooled_sd:
                row["z_ref_to_path"] = (path_min - ref) / pooled_sd
            else:
                row["z_ref_to_path"] = np.nan
        rows.append(row)

    return pd.DataFrame(rows)


df = build_dataframe()
print(f"{len(df)} loci loaded, {df['pathogenic_min'].notna().sum()} with pathogenic thresholds.")
df.head()


63 loci loaded, 63 with pathogenic thresholds.


,gene,disease_id,disease,locus_id,motif,motif_len_bp,ref_repeats,region_catalog,region_class,region_strchive,...,intermediate_max,pathogenic_min,pathogenic_max,total_allele_n,pooled_pop_sd,cohort_used,per_cohort_sd,fold_change,delta_units,z_ref_to_path
0,COMP,EDM1-PSACH_COMP,"Multiple epiphyseal dysplasia, Pseudoachondrop...",19-18786034-18786049-GTC,GTC,3,5,CDS,coding,Coding Exon 13,...,NaN,6,7,512,0.000000,HPRC256,{'HPRC256': 0.0},1.200000,1,NaN
1,ARX,PRTS_ARX,Partington syndrome,X-25013529-25013565-NGC,NGC,3,12,CDS,coding,"Coding Exon 2, aa 144-155",...,NaN,20,20,393,0.000000,HPRC256,{'HPRC256': 0.0},1.666667,8,NaN
2,HOXA13,HFG_HOXA13-II,Hand-foot-genital syndrome 2,7-27199825-27199861-NGC,NGC,3,12,CDS,coding,Coding Exon 1,...,NaN,18,18,512,0.000000,HPRC256,{'HPRC256': 0.0},1.500000,6,NaN
3,RUNX2,CCD_RUNX2,Cleidocranial dysplasia,6-45422750-45422792-GCN,GCN,3,14,CDS,coding,Coding Exon 3,...,NaN,20,27,668,0.192835,HPRC256+T2T,"{'HPRC256': 0.0, 'T2T': 0.399}",1.428571,6,31.114722
4,FOXL2,BPES_FOXL2,"Blepharophimosis, epicanthus inversus, and ptosis",3-138946020-138946062-NGC,NGC,3,14,CDS,coding,Coding Exon 1,...,NaN,15,24,512,0.000000,HPRC256,{'HPRC256': 0.0},1.071429,1,NaN


## 2. Interactive visualizer

Color = coarse region class (coding / noncoding / ambiguous, colorblind-safe at this count).
Marker shape = fine-grained region (CDS, 5' UTR, 3' UTR, intron, ambiguous/isoform-dependent).
Hover any point for the full detail (gene, disease, thresholds, SD, evidence tier).

**On the z-score view specifically:** 8 loci (all CDS polyalanine tracts — COMP, ARX×2,
FOXL2, HOXA13×2, SOX3, ZIC2) show literally zero length variation across every haplotype in
every available cohort, so `pathogenic distance ÷ population SD` is undefined — not just
large. Rather than drop them, they're plotted as **open/hollow markers** at a fixed height
above a dotted divider line, so you can still see they're the tightest-population loci in
the set without implying a specific z-value. They're always classified `far` under this
metric (undefined reads as the extreme end, by construction) — worth noticing this
contradicts their `close` classification under fold-change (their pathogenic thresholds are
only 1.06–1.67x reference, the smallest fold-changes in the whole dataset). Both are
"correct" — it's a real, informative artifact of what z-score means when the denominator is
immeasurably small, not a bug.


In [3]:
VWA1_GENE = "VWA1"  # deletion mechanism, not an expansion — excluded from fold/z framing

COARSE_COLORS = {
    "coding": "#2a78d6",
    "noncoding": "#eb6834",
    "ambiguous": "#1baf7a",
}
COARSE_LABELS = {
    "coding": "Coding (CDS)",
    "noncoding": "Noncoding (UTR / intron)",
    "ambiguous": "Ambiguous / isoform-dependent",
}
FINE_SYMBOLS = {
    "CDS": "circle",
    "5' UTR": "triangle-down",
    "3' UTR": "triangle-up",
    "intron": "square",
    "exon": "diamond",
}
DEFAULT_SYMBOL = "circle"

INHERITANCE_TAGS = ["AD", "AR", "XR", "XD"]
EVIDENCE_TAGS = ["Definitive", "Strong", "Moderate", "Limited", "Disputed"]

METRIC_INFO = {
    "fold_change": dict(label="Fold-change (pathogenic ÷ reference)", default_cutoff=3.0,
                          slider_min=1.0, slider_max=200.0, log_axis=True),
    "z_ref_to_path": dict(label="z-score (pathogenic distance ÷ population SD)", default_cutoff=15.0,
                            slider_min=0.5, slider_max=650.0, log_axis=True),
}

base_df = df[df["gene"] != VWA1_GENE].copy()

# ---- widgets -----------------------------------------------------------
sd_cutoff_w = widgets.FloatSlider(value=5.0, min=0.0, max=35.0, step=0.25,
                                   description="Population SD cutoff", style={"description_width": "160px"},
                                   layout=widgets.Layout(width="480px"), continuous_update=False)

metric_w = widgets.ToggleButtons(options=[("Fold-change", "fold_change"), ("z-score", "z_ref_to_path")],
                                  description="Distance metric", style={"description_width": "160px"})

dist_cutoff_w = widgets.FloatLogSlider(value=3.0, base=10, min=0, max=np.log10(650), step=0.01,
                                        description="Pathogenic-distance cutoff",
                                        style={"description_width": "160px"},
                                        layout=widgets.Layout(width="480px"), continuous_update=False)

evidence_w = widgets.SelectMultiple(options=EVIDENCE_TAGS, value=tuple(EVIDENCE_TAGS),
                                     description="Evidence tier", rows=5,
                                     style={"description_width": "160px"})

inheritance_w = widgets.SelectMultiple(options=INHERITANCE_TAGS, value=tuple(INHERITANCE_TAGS),
                                        description="Inheritance", rows=4,
                                        style={"description_width": "160px"})

exclude_vwa1_note = widgets.HTML("<i>VWA1 (deletion mechanism) is always excluded from this view.</i>")

stats_out = widgets.Output()
plot_out = widgets.Output()


def hover_text(r, metric):
    if r.get("is_sentinel", False):
        return (f"<b>{r['gene']}</b> — {r['disease']}<br>"
                f"region: {r['region_strchive']} ({r['region_class']})<br>"
                f"ref={r['ref_repeats']}  benign_max={r['benign_max']}  pathogenic_min={r['pathogenic_min']}<br>"
                f"population SD=0 across {r['total_allele_n']} alleles ({r['cohort_used']}) → z UNDEFINED<br>"
                f"<i>plotted at top edge as an unbounded outlier, not a measured value</i><br>"
                f"(fold-change is only {r['fold_change']:.2f}× — modest in absolute terms;<br>"
                f"it's only \"far\" in z-units because there's no measurable denominator)<br>"
                f"evidence={r['evidence']}  inheritance={','.join(r['inheritance'])}")
    dist_val = r[metric]
    dist_label = "fold" if metric == "fold_change" else "z"
    return (f"<b>{r['gene']}</b> — {r['disease']}<br>"
            f"region: {r['region_strchive']} ({r['region_class']})<br>"
            f"ref={r['ref_repeats']}  benign_max={r['benign_max']}  pathogenic_min={r['pathogenic_min']}<br>"
            f"population SD={r['pooled_pop_sd']:.2f}  {dist_label}={dist_val:.2f}<br>"
            f"evidence={r['evidence']}  inheritance={','.join(r['inheritance'])}")


def make_figure(sub, metric, info, sd_cut, dist_cut, counts, sentinel_boundary=None):
    # Plain go.Figure, rebuilt on every change and redisplayed — avoids go.FigureWidget,
    # which needs the `anywidget` package and has shown version-compatibility breakage
    # (AttributeError on DOMWidget) across some ipywidgets/plotly/Jupyter combinations.
    fig = go.Figure()
    for cls in ["noncoding", "coding", "ambiguous"]:
        csub = sub[sub["region_class"] == cls]
        symbols = csub["region_catalog"].map(FINE_SYMBOLS).fillna(DEFAULT_SYMBOL)
        symbols = symbols.where(~csub["is_sentinel"], symbols + "-open")
        sizes = np.where(csub["is_sentinel"], 13, 11)
        texts = [hover_text(r, metric) for _, r in csub.iterrows()]
        fig.add_trace(go.Scatter(
            x=csub["_x"], y=csub["_y"], mode="markers", name=COARSE_LABELS[cls],
            marker=dict(color=COARSE_COLORS[cls], size=sizes, symbol=symbols,
                        line=dict(color="white", width=1)),
            text=texts, hovertemplate="%{text}<extra></extra>",
        ))

    shapes = [
        dict(type="line", xref="x", yref="paper", x0=sd_cut, x1=sd_cut, y0=0, y1=1,
             line=dict(color="#8a8a86", width=1, dash="dash")),
        dict(type="line", xref="paper", yref="y", x0=0, x1=1, y0=dist_cut, y1=dist_cut,
             line=dict(color="#8a8a86", width=1, dash="dash")),
    ]
    annotations = [
        dict(x=0.02, y=0.98, xref="paper", yref="paper", showarrow=False, xanchor="left", yanchor="top",
             text=f"tight · far  (n={counts.get('tight_far', 0)})", font=dict(size=11, color="#52514e")),
        dict(x=0.98, y=0.98, xref="paper", yref="paper", showarrow=False, xanchor="right", yanchor="top",
             text=f"spread · far  (n={counts.get('spread_far', 0)})", font=dict(size=11, color="#52514e")),
        dict(x=0.02, y=0.02, xref="paper", yref="paper", showarrow=False, xanchor="left", yanchor="bottom",
             text=f"tight · close  (n={counts.get('tight_close', 0)})", font=dict(size=11, color="#52514e")),
        dict(x=0.98, y=0.02, xref="paper", yref="paper", showarrow=False, xanchor="right", yanchor="bottom",
             text=f"spread · close  (n={counts.get('spread_close', 0)})", font=dict(size=11, color="#52514e")),
    ]

    if sentinel_boundary is not None:
        shapes.append(dict(type="line", xref="paper", yref="y", x0=0, x1=1,
                            y0=sentinel_boundary, y1=sentinel_boundary,
                            line=dict(color="#c3c2b7", width=1, dash="dot")))
        annotations.append(dict(
            xref="paper", yref="y", x=0.5, y=sentinel_boundary, xanchor="center", yanchor="bottom",
            showarrow=False,
            text="above this line: population SD = 0 in every cohort — z undefined, plotted as unbounded (open markers)",
            font=dict(size=10, color="#8a8a86"),
        ))

    fig.update_layout(
        template="plotly_white",
        height=650,
        title="Population tightness x pathogenic distance",
        xaxis_title="Population SD of repeat copy number (repeat units)",
        yaxis_title=info["label"],
        yaxis_type="log" if info["log_axis"] else "linear",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        margin=dict(t=90, r=40, l=70, b=60),
        shapes=shapes,
        annotations=annotations,
    )
    return fig


def update(*_):
    metric = metric_w.value
    info = METRIC_INFO[metric]

    sub = base_df.copy()
    sub = sub[sub["evidence"].isin(evidence_w.value)]
    sub = sub[sub["inheritance"].apply(lambda tags: any(t in inheritance_w.value for t in tags))]
    sub = sub[sub["pathogenic_min"].notna()]
    # NB: for metric="z_ref_to_path", SD=0 loci have z=NaN by construction (undefined, not
    # missing-data) — don't drop them here, the sentinel branch below handles them.
    if metric == "fold_change":
        sub = sub[sub[metric].notna()]

    sd_cut = sd_cutoff_w.value
    dist_cut = dist_cutoff_w.value
    sentinel_boundary = None

    if metric == "fold_change":
        sub = sub.assign(_x=sub["pooled_pop_sd"], _y=sub["fold_change"], is_sentinel=False)
    else:
        zero_mask = sub["pooled_pop_sd"] == 0
        finite = sub[~zero_mask].assign(_x=sub.loc[~zero_mask, "pooled_pop_sd"],
                                         _y=sub.loc[~zero_mask, "z_ref_to_path"],
                                         is_sentinel=False)
        zero_rows = sub[zero_mask].copy()
        if len(zero_rows):
            # Visual sentinel, not a statistical estimate: these loci show literally zero
            # length variation across every haplotype in every available cohort, so z
            # (pathogenic distance / population SD) is undefined, not merely large. Rather
            # than drop them, plot them as open markers at a fixed height above the real
            # data, with a dotted divider making clear that height carries no numeric meaning.
            finite_max = finite["_y"].max() if len(finite) else dist_cut
            sentinel_y = finite_max * 4
            sentinel_boundary = finite_max * 2
            zero_rows["_x"] = zero_rows["pooled_pop_sd"]
            zero_rows["_y"] = sentinel_y
            zero_rows["is_sentinel"] = True
        sub = pd.concat([finite, zero_rows], ignore_index=False) if len(zero_rows) else finite

    sub["pop_group"] = np.where(sub["pooled_pop_sd"] < sd_cut, "tight", "spread")
    if metric == "z_ref_to_path":
        # Sentinel rows are always "far": with z undefined (not just large), calling them
        # anything but the extreme end of this metric would be arbitrary. This is a direct,
        # sometimes counterintuitive, consequence of using z rather than fold-change as the
        # distance metric — see the markdown note below the plot.
        sub["dist_group"] = np.where(sub["is_sentinel"], "far", np.where(sub[metric] < dist_cut, "close", "far"))
    else:
        sub["dist_group"] = np.where(sub[metric] < dist_cut, "close", "far")
    sub["quadrant"] = sub["pop_group"] + "_" + sub["dist_group"]
    counts = sub["quadrant"].value_counts()

    fig = make_figure(sub, metric, info, sd_cut, dist_cut, counts, sentinel_boundary)
    plot_out.clear_output(wait=True)
    with plot_out:
        display(fig)

    stats_out.clear_output(wait=True)
    with stats_out:
        print(f"n={len(sub)} loci shown (of {len(base_df)} total, VWA1 excluded)")
        print()
        ct = pd.crosstab(sub["quadrant"], sub["region_class"])
        print("Quadrant x region class:")
        print(ct.to_string())
        print()

        ct2_src = sub[sub["region_class"] != "ambiguous"]
        ct2 = pd.crosstab(ct2_src["dist_group"], ct2_src["region_class"])
        if ct2.shape == (2, 2):
            odds, p = stats.fisher_exact(ct2)
            print(f"Fisher's exact (coding vs noncoding x close vs far): OR={odds:.2f}, p={p:.2e}")

        n_sentinel = int(sub["is_sentinel"].sum()) if "is_sentinel" in sub else 0
        coding = sub[(sub["region_class"] == "coding") & (~sub["is_sentinel"])][metric].dropna()
        noncoding = sub[(sub["region_class"] == "noncoding") & (~sub["is_sentinel"])][metric].dropna()
        if len(coding) > 1 and len(noncoding) > 1:
            u, p = stats.mannwhitneyu(coding, noncoding)
            note = f"  ({n_sentinel} SD=0 sentinel loci excluded — z undefined, not a number)" if n_sentinel else ""
            print(f"Mann-Whitney U on {metric}, coding vs noncoding: p={p:.2e}  "
                  f"(coding median={coding.median():.2f}, noncoding median={noncoding.median():.2f}){note}")
        print()
        print("Genes per quadrant:")
        for q, qsub in sub.groupby("quadrant"):
            tags = qsub["gene"] + qsub["is_sentinel"].map({True: " (SD=0, sentinel)", False: ""})
            print(f"  {q} (n={len(qsub)}): " + ", ".join(sorted(tags)))


for w in [sd_cutoff_w, dist_cutoff_w, metric_w, evidence_w, inheritance_w]:
    w.observe(update, names="value")


def on_metric_change(change):
    info = METRIC_INFO[change["new"]]
    dist_cutoff_w.min = np.log10(info["slider_min"])
    dist_cutoff_w.max = np.log10(info["slider_max"])
    dist_cutoff_w.value = info["default_cutoff"]


metric_w.observe(on_metric_change, names="value")

update()

controls = widgets.VBox([
    widgets.HBox([sd_cutoff_w, dist_cutoff_w]),
    metric_w,
    widgets.HBox([evidence_w, inheritance_w]),
    exclude_vwa1_note,
])

display(controls, plot_out, stats_out)


Output()

Output()

## 3. Export the current view

Re-run this cell any time to write the table currently selected by the widgets above
(same filters, same quadrant assignment) to `current_view.csv`.


In [4]:
def current_view():
    metric = metric_w.value
    sub = base_df.copy()
    sub = sub[sub["evidence"].isin(evidence_w.value)]
    sub = sub[sub["inheritance"].apply(lambda tags: any(t in inheritance_w.value for t in tags))]
    sub = sub[sub["pathogenic_min"].notna()]
    if metric == "fold_change":
        sub = sub[sub[metric].notna()]
    sub["is_sentinel"] = (metric == "z_ref_to_path") & (sub["pooled_pop_sd"] == 0)
    sub["pop_group"] = np.where(sub["pooled_pop_sd"] < sd_cutoff_w.value, "tight", "spread")
    if metric == "z_ref_to_path":
        sub["dist_group"] = np.where(sub["is_sentinel"], "far",
                                      np.where(sub[metric] < dist_cutoff_w.value, "close", "far"))
    else:
        sub["dist_group"] = np.where(sub[metric] < dist_cutoff_w.value, "close", "far")
    sub["quadrant"] = sub["pop_group"] + "_" + sub["dist_group"]
    return sub.drop(columns=["per_cohort_sd"]).sort_values(["quadrant", "gene"])


view = current_view()
view.to_csv("current_view.csv", index=False)
print(f"Wrote current_view.csv ({len(view)} rows).")
view


Wrote current_view.csv (62 rows).


,gene,disease_id,disease,locus_id,motif,motif_len_bp,ref_repeats,region_catalog,region_class,region_strchive,...,total_allele_n,pooled_pop_sd,cohort_used,fold_change,delta_units,z_ref_to_path,is_sentinel,pop_group,dist_group,quadrant
58,DAB1,SCA37_DAB1,Spinocerebellar ataxia type 37,1-57367043-57367118-AAAAT,AAAAT,5,15,intron,noncoding,Intron 1 (most isoforms),...,9526,17.120855,HPRC256+Illumina174k+T2T+TenK10K,2.066667,16,0.934533,False,spread,close,spread_close
47,DMPK,DM1_DMPK,Myotonic dystrophy type 1,19-45770204-45770264-CAG,CAG,3,20,3' UTR,noncoding,3' UTR,...,4518,5.595419,HPRC256+T2T+TenK10K,2.500000,30,5.361529,False,spread,close,spread_close
52,TCF4,FECD3_TCF4,Fuchs endothelial corneal dystrophy 3,18-55586155-55586227-CAG,CAG,3,24,5' UTR,noncoding,Intron 1,...,4518,24.283324,HPRC256+T2T+TenK10K,2.125000,27,1.111874,False,spread,close,spread_close
61,ATXN10,SCA10_ATXN10,Spinocerebellar ataxia type 10,22-45795354-45795424-ATTCT,ATTCT,5,14,intron,noncoding,Intron 9,...,4518,30.589241,HPRC256+T2T+TenK10K,57.142857,786,25.695309,False,spread,far,spread_far
60,ATXN8OS,SCA8_ATXN8OS,Spinocerebellar ataxia type 8,13-70139383-70139428-CTG,CTG,3,15,exon,ambiguous,"Coding Exon 1, or 3' UTR depending on transcript",...,4518,21.825635,HPRC256+T2T+TenK10K,4.733333,56,2.565790,False,spread,far,spread_far
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31,NUTM2B-AS1,OPML1_NUTM2B-AS1,Oculopharyngeal myopathy with leukoencephalopa...,10-79826383-79826404-GGC,GGC,3,7,exon,ambiguous,Exon 1 of lncRNA (noncoding),...,9510,2.102124,HPRC256+Illumina174k+T2T+TenK10K,23.000000,154,73.259239,False,tight,far,tight_far
34,PPP2R2B,SCA12_PPP2R2B,Spinocerebellar ataxia type 12,5-146878727-146878757-GCT,GCT,3,10,5' UTR,noncoding,5' UTR,...,4518,2.338751,HPRC256+T2T+TenK10K,5.100000,41,17.530726,False,tight,far,tight_far
32,RAPGEF2,FAME7_RAPGEF2,Familial adult myoclonic epilepsy type 7,4-159342526-159342616-TTTTA,TTTTA,5,18,intron,noncoding,Intron 14,...,9526,2.201698,HPRC256+Illumina174k+T2T+TenK10K,3.333333,42,19.076186,False,tight,far,tight_far
16,RILPL1,OPDM4_RILPL1,Oculopharyngodistal myopathy type 4,12-123533720-123533750-GGC,GGC,3,10,5' UTR,noncoding,5' UTR,...,9526,0.469573,HPRC256+Illumina174k+T2T+TenK10K,12.000000,110,234.255145,False,tight,far,tight_far


## Caveats (carried over from the original analysis)

- STRchive evidence tiers in this catalog: 42 Definitive, 7 Moderate, 3 Strong, 9 Limited,
  2 Disputed. Use the evidence-tier filter above to check robustness.
- C9orf72's STRchive `pathogenic_min` (31) is the low end of a widely-used gray zone (24-30);
  most labs only call clear pathogenicity at ≥60-100+ repeats. Using a stricter threshold
  would push it further into "far/outlier" territory, not change its category.
- Population "reference mean" = the reference-genome haploid copy number, not a cohort mean —
  simplification, but the reference falls inside the benign range for every locus checked, and
  it sidesteps an encoding ambiguity in a few polyalanine loci's allele histograms (their SDs are
  unaffected, since SD is shift-invariant).
- `z_ref_to_path` is undefined (and excluded) wherever a locus's population SD is 0 across every
  available cohort — mostly the polyalanine loci that are essentially fixed-length in a few hundred
  people. That's a real finding (their pathogenic distance is "however many SDs you like" for
  data-availability reasons), not a bug to fix.
- Cutoff defaults (SD=5, fold=3x) are empirical gaps in this specific 63-locus catalog, not
  general truths about tandem repeats — that's exactly why they're sliders here rather than
  constants.
